# Module 07 — AI Workbook Companion (GPU-accelerated NN inference)

A lightweight companion to
[cuda_neural_network_demo_complete.ipynb](../cuda_neural_network_demo_complete.ipynb).
It does **not** replace it — it adds the supervision layer from
[Module 11](../../11/README.md) to Module 07's network: the Allen-style
single-hidden-layer classifier (**4 inputs → 8 ReLU → 1 sigmoid**), run over a
large **batch of events**.

New here? Read [Module 11's README](../../11/README.md) first. The failure mode
here is one from the README list: an **"optimization" that changes the math**. The
AI makes the GPU version *fast* — but subtly wrong — and it still returns the right
shape with every output a valid probability in `[0, 1]`, so it slips past a shape
check.

## The task: fast batched inference, and prove it's still correct

The main notebook runs this network one event per CUDA thread. Here you run the
**same network** over millions of events at once with array operations (NumPy on
the CPU, **CuPy** on the GPU). The goal is the GPU speedup — but it only counts if
the answer still matches a trusted CPU reference.

The five phases (you own all but GENERATE):

1. **SPECIFY** — Inputs `(N, 4)` float32; params: per-feature `mean`/`istd`,
   `W1 (8,4)`, `b1 (8)`, `W2 (8)`, `b2`. Output `(N,)` in `[0, 1]`. Metrics:
   speedup vs CPU **and** max-abs-error vs the reference.
2. **GENERATE** — Ask the AI for a fast batched GPU version. Keep it separate.
3. **PREDICT** — *Before running:* where can a "GPU optimization" quietly change
   the math? (Hint: how are the inputs normalised?)
4. **VERIFY** — [verify_fcnn.py](./verify_fcnn.py) has a CPU reference, a
   PASS/FAIL gate (max-abs-error ≤ tol), and a GPU-vs-CPU speedup report.
5. **DIAGNOSE** — Explain why the fast version still produced valid-looking
   probabilities yet failed the gate.

## The adversarial exercise

[adversarial_fcnn_buggy.py](./adversarial_fcnn_buggy.py) is the network "an AI
optimized for the GPU." To skip uploading the trained `mean`/`std`, it normalises
each batch with **the batch's own statistics** (`X.mean(axis=0)`,
`X.std(axis=0)`) instead of the model's fixed per-feature values. It is fast and
shape-preserving — and wrong, because inference must use the trained
normalisation, not whatever the current batch happens to look like.

Your job:

1. Predict where the math changed *before* running.
2. Run it, then run the verifier and read the max-abs-error **and** the speedup.
3. State the exact bug (batch statistics vs the model's fixed `mean`/`istd`) and
   the fix.
4. Prove the fix matches the reference — while keeping the speedup.

Could an AI catch this? Sometimes — but "it's faster and the outputs look like
probabilities" passes a casual review. Comparing against a CPU reference catches
it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why did normalising with batch statistics still produce values in [0,1] and the
  right shape, yet the wrong answer? What single check exposes it?
- "Fast" and "correct" are separate questions. Which printed number answered each?
- On the GPU, what is the cheapest correct way to apply the trained, fixed
  per-feature normalisation to a whole batch?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Run the problem program [adversarial_fcnn_buggy.py](./adversarial_fcnn_buggy.py). Read it first and predict what will happen before you run this cell.


In [ ]:
!python adversarial_fcnn_buggy.py

## Step 2 - Run your verification harness

[verify_fcnn.py](./verify_fcnn.py) ships a CPU reference, a PASS/FAIL gate, and a GPU-vs-CPU speedup report, but it **defaults to FAIL** until you fix the function under test. Edit the file, then re-run this cell until it prints `PASS`.


In [ ]:
!python verify_fcnn.py

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.